# 00C · BEV、Occupancy 与 Scene Representation：为什么自动驾驶不只做 2D 检测？

你已经知道卷积、attention、feature map 等深度学习概念。本节补充的是自动驾驶中的表示选择：同一段道路可以表示为 camera image、LiDAR points、3D boxes、BEV occupancy、lane graph、vectorized agents 或 map elements。

这里的关键问题不是“哪个模型更大”，而是：**下游 prediction/planning 需要什么几何和拓扑信息？** BEV 是把多视角/多传感器信息放到统一的地面坐标参考中，方便进行空间关系、速度和地图约束建模。

本节用一个小型点云 rasterizer 直观比较 resolution、dropout 和 occupancy IoU；`06` 再深入 LiDAR/BEV occupancy，`02` 处理多传感器 feature fusion。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(9)
vehicle = np.c_[rng.normal(16.0, 0.9, 280), rng.normal(0.0, 0.7, 280)]
pedestrian = np.c_[rng.normal(9.0, 0.25, 45), rng.normal(-3.2, 0.25, 45)]
lane_marking = np.c_[np.linspace(0, 30, 150), np.full(150, 3.5)]
points = np.vstack([vehicle, pedestrian, lane_marking])

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(points[:, 0], points[:, 1], s=4, alpha=0.45)
ax.set_aspect("equal")
ax.set(xlabel="forward x / m", ylabel="left y / m", title="A sparse scene before choosing a representation")


In [ ]:
def rasterize(points_xy, resolution=0.5, x_range=(0, 32), y_range=(-8, 8)):
    x_edges = np.arange(x_range[0], x_range[1] + resolution, resolution)
    y_edges = np.arange(y_range[0], y_range[1] + resolution, resolution)
    occupancy, _, _ = np.histogram2d(points_xy[:, 0], points_xy[:, 1], bins=[x_edges, y_edges])
    return (occupancy > 0).astype(np.uint8), x_edges, y_edges

occupancy, x_edges, y_edges = rasterize(points, resolution=0.5)
plt.figure(figsize=(8, 4))
plt.imshow(occupancy.T, origin="lower", aspect="auto", extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]])
plt.xlabel("forward x / m")
plt.ylabel("left y / m")
plt.title(f"BEV occupancy: grid={occupancy.shape}, occupied cells={occupancy.sum()}")
plt.colorbar(label="occupied")


In [ ]:
from ipywidgets import FloatSlider, interact

def occupancy_experiment(resolution=0.5, dropout=0.0):
    keep = np.random.default_rng(20).random(len(points)) > dropout
    estimate, _, _ = rasterize(points[keep], resolution=resolution)
    target, _, _ = rasterize(points, resolution=resolution)
    intersection = np.logical_and(estimate, target).sum()
    union = np.logical_or(estimate, target).sum()
    iou = intersection / max(union, 1)
    print(f"resolution={resolution:.2f} m, dropout={dropout:.2f}, occupancy IoU={iou:.3f}")
    print("representation question: occupancy preserves space; vector/map representations preserve semantics and topology")

interact(
    occupancy_experiment,
    resolution=FloatSlider(min=0.2, max=1.5, step=0.1, value=0.5, description="grid / m"),
    dropout=FloatSlider(min=0, max=0.8, step=0.05, value=0.0, description="dropout"),
)


## 领域检查点

1. occupancy、3D box、lane vector、agent state 各自保留了什么信息，又丢掉了什么信息？
2. 为什么 BEV 对 planning 友好，但不能因此认为“所有任务都应该转成 BEV”？
3. 如果 occupancy IoU 不变，车辆速度或时间戳错位仍可能让 prediction 失败吗？为什么？

**下一步**：`02` 研究 camera/LiDAR feature fusion；`06` 研究点云体素化和 occupancy；两者都建立在这里的表示选择之上。
